Test flat RL without action masking

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.stack_gym import StackEnv
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback
from sb3_contrib.common.maskable.utils import get_action_masks




# Configuration
config_dict = {
    "vessel_shape": (3, 2, 2),
    "yard_shape": (3, 2, 2),
    "num_containers": 12,
    "group_num": 3,
    "group_placement": "random",
    "seed": 4307,
    "reward_scheme": "simple_single_stack",
    "observation_type": "stack_features"
}

# Create environment and agent
env = StackEnv(config=config_dict,  render_mode="rgb_array")

# Initialize PPO
print("=" * 70)
print("TRAINING PPO WITHOUT ACTION MASKING ON STACKENV")
print("=" * 70)

policy_kwargs = dict(net_arch=[256, 256, 32])
model = PPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 100000 timesteps...")
model.learn(total_timesteps=100_000)
print("Training complete")


# Save model
model.save("ppo_stack_flat")


In [ ]:

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0

for step in range(12):

    action, _states = model.predict(obs)
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(12):
    action, _states = model.predict(obs)
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

Test flat RL with action masking

Flat RL with new obs space on simple reward scheme

In [ ]:
import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback



# Configuration
config_dict = {
    "vessel_shape": (3, 3, 3),
    "yard_shape": (3, 3, 3),
    "num_containers": 27,
    "group_num": 3,
    "group_placement": "random",
    "seed": 4307,
    "reward_scheme": "simple_single_stack",
    "observation_type": "stack_features"
}

# Create environment and agent
env = StackEnv(config=config_dict,  render_mode="rgb_array")



policy_kwargs = dict(net_arch=[256, 256, 32])
model = MaskablePPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 1500000 timesteps")
#model.learn(total_timesteps=500_000, callback = callback)
model.learn(total_timesteps=1_500_000)
print("Training complete")


# Save model
model.save("ppo_stack_flat_masked_1.5M_simple_reward")
print("Model saved to ppo_stack_flat_masked_1.5M_simple_reward")

In [ ]:

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0

for step in range(100):

    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(64):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

Training Flat RL agent on complex reward scheme

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback



# Configuration
config_dict = {
    "vessel_shape": (3, 3, 3),
    "yard_shape": (3, 3, 3),
    "num_containers": 27,
    "group_num": 3,
    "group_placement": "random",
    "seed": 4307,
    #"reward_scheme": "simple_single_stack",
    "observation_type": "stack_features"
}

# Create environment and agent
env = StackEnv(config=config_dict,  render_mode="rgb_array")



policy_kwargs = dict(net_arch=[256, 256, 32])
model = MaskablePPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 1500000 timesteps")
#model.learn(total_timesteps=500_000, callback = callback)
model.learn(total_timesteps=1_500_000)
print("Training complete")


# Save model
model.save("ppo_stack_flat_masked_1.5M_complex_reward")
print("Model saved to ppo_stack_flat_masked_1.5M_complex_reward")

In [ ]:
from sb3_contrib.common.maskable.utils import get_action_masks

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0

for step in range(60):

    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)